# Binance Cross-Margin: Live Inventory + Next-Hourly-Interest

Written by the daemon `binance_margin_inventory_hourly_interest_rate_to_parquet`.

Path: `$TRADE_DATA/binance/spot/margin_inventory_hourly_interest/margin_inv_hourly_{YYYY-MM-DD}.parquet`
(one file per UTC day; rotated to `margin_inv_hourly_{date}.{ts}.parquet` when >10 MB).

Columns: `ts, symbol, inventory, next_hourly_interest_rate`

In [ ]:
import os
from pathlib import Path
import pandas as pd

pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 160)

# --- config ---
SYMBOL = "STORJ"  # symbol to drill into
N = 30            # number of latest rows to show

TRADE_DATA = os.environ.get("TRADE_DATA", "/ndata/trade/data")
inv_base = Path(TRADE_DATA) / "binance" / "spot" / "margin_inventory_hourly_interest"
print("Base:", inv_base)

In [ ]:
# Loads every daily/rotated file so the full series is available
inv_files = sorted(inv_base.glob("margin_inv_hourly_*.parquet"))
print(f"{len(inv_files)} file(s):")
for f in inv_files:
    print("  ", f.name, f"({f.stat().st_size/1e6:.2f} MB)")

inv = pd.concat((pd.read_parquet(f) for f in inv_files), ignore_index=True)
inv["datetime"] = pd.to_datetime(inv["ts"], unit="ms")
inv = inv.sort_values("ts").reset_index(drop=True)
print(
    f"\n{len(inv):,} rows | {inv['symbol'].nunique():,} symbols | "
    f"{inv['datetime'].min()} .. {inv['datetime'].max()}"
)

In [ ]:
# Latest tick snapshot across all symbols (top inventory first)
latest_ts = inv["ts"].max()
snap = (
    inv[inv["ts"] == latest_ts]
    .sort_values("inventory", ascending=False)
    .reset_index(drop=True)
)
print(f"Latest tick: {pd.to_datetime(latest_ts, unit='ms')} ({len(snap):,} symbols)")
snap.head(N)

In [ ]:
# Time series for one SYMBOL
sym_df = inv[inv["symbol"] == SYMBOL].reset_index(drop=True)
print(f"{SYMBOL}: {len(sym_df):,} ticks")
sym_df[["datetime", "symbol", "inventory", "next_hourly_interest_rate"]].tail(N)